# บทเรียนเกม Sudoku พื้นฐาน (เวอร์ชันปรับปรุง)

ในบทเรียนนี้ เราจะเรียนรู้วิธีสร้างเกม Sudoku พื้นฐานในภาษา Python พร้อมฟังก์ชันต่าง ๆ ที่มีประโยชน์

## ส่วนที่ 1: ความเข้าใจเกี่ยวกับ Sudoku

Sudoku คือเกมปริศนาตัวเลขที่:
- ประกอบด้วยตารางขนาด 9x9
- แบ่งออกเป็นกล่อง 3x3 จำนวน 9 กล่อง
- แต่ละแถว แต่ละคอลัมน์ และแต่ละกล่อง 3x3 ต้องมีตัวเลข 1-9 เพียงครั้งเดียว

### วิธีการแก้ปริศนา
เราจะใช้วิธี **Backtracking** ซึ่งเป็นวิธีค้นหาแบบเจาะลึก (Depth-First Search):
1. ค้นหาช่องว่าง (0)
2. ลองใส่ตัวเลข 1-9
3. ถ้าตัวเลขที่ใส่ไม่ขัดแย้ง ให้ดำเนินการต่อ
4. ถ้าไม่สามารถแก้ได้ ให้ลบตัวเลขและลองตัวเลขถัดไป

## ส่วนที่ 2: สร้างคลาส Sudoku พร้อมฟังก์ชันต่าง ๆ

In [ ]:
class SudokuGame:
    def __init__(self, board=None):
        """
        สร้างเกม Sudoku ใหม่
        
        พารามิเตอร์:
        - board: ตารางเกม (ถ้าไม่ระบุจะใช้ค่าเริ่มต้น)
        """
        # ตารางเกม Sudoku (0 = ช่องว่าง)
        if board is None:
            self.board = [
                [5, 3, 0, 0, 7, 0, 0, 0, 0],
                [6, 0, 0, 1, 9, 5, 0, 0, 0],
                [0, 9, 8, 0, 0, 0, 0, 6, 0],
                [8, 0, 0, 0, 6, 0, 0, 0, 3],
                [4, 0, 0, 8, 0, 3, 0, 0, 1],
                [7, 0, 0, 0, 2, 0, 0, 0, 6],
                [0, 6, 0, 0, 0, 0, 2, 8, 0],
                [0, 0, 0, 4, 1, 9, 0, 0, 5],
                [0, 0, 0, 0, 8, 0, 0, 7, 9]
            ]
        else:
            self.board = [row[:] for row in board]  # สำเนาตารางเพื่อไม่ให้เปลี่ยนต้นฉบับ
        
        # เก็บตารางเดิมเพื่อให้สามารถรีเซ็ตได้
        self.initial_board = [row[:] for row in self.board]
        self.moves = []  # บันทึกการเดินของผู้เล่น
    
    def display_board(self):
        """
        แสดงตารางเกม Sudoku อย่างสวยงาม
        """
        print()
        for i in range(9):
            if i % 3 == 0 and i != 0:
                print("-" * 25)
            
            row = ""
            for j in range(9):
                if j % 3 == 0 and j != 0:
                    row += "| "
                
                value = self.board[i][j]
                row += (str(value) + " ") if value != 0 else ". "
            
            print(row)
        print()
    
    def is_valid(self, row, col, num):
        """
        ตรวจสอบว่าสามารถวางตัวเลขได้หรือไม่
        
        พารามิเตอร์:
        - row: แถว (0-8)
        - col: คอลัมน์ (0-8)
        - num: ตัวเลข (1-9)
        
        ตรวจสอบ 3 เงื่อนไข:
        1. ไม่มีตัวเลขเดียวกันในแถว
        2. ไม่มีตัวเลขเดียวกันในคอลัมน์
        3. ไม่มีตัวเลขเดียวกันในกล่อง 3x3
        """
        
        # ตรวจสอบแถว
        if num in self.board[row]:
            return False
        
        # ตรวจสอบคอลัมน์
        for i in range(9):
            if self.board[i][col] == num:
                return False
        
        # ตรวจสอบกล่อง 3x3
        # คำนวณมุมซ้ายบนของกล่อง 3x3 ที่ตำแหน่งปัจจุบัน
        box_row = (row // 3) * 3
        box_col = (col // 3) * 3
        
        for i in range(box_row, box_row + 3):
            for j in range(box_col, box_col + 3):
                if self.board[i][j] == num:
                    return False
        
        return True
    
    def solve(self):
        """
        แก้ไขปริศนา Sudoku โดยใช้วิธี Backtracking
        
        ขั้นตอน:
        1. วนลูปทั้งตาราง 9x9
        2. เมื่อพบช่องว่าง (0) ให้ลองใส่ตัวเลข 1-9
        3. ถ้าตัวเลขถูกต้อง ให้เรียก solve() แบบเรียกซ้ำ
        4. ถ้าไม่สำเร็จ ให้คืนค่าเป็น 0 และลองตัวเลขถัดไป
        5. ถ้าไม่มีตัวเลขไหนใช้ได้ ให้คืนค่า False (Backtrack)
        """
        for row in range(9):
            for col in range(9):
                # หาช่องว่างแรก
                if self.board[row][col] == 0:
                    # ลองใส่ตัวเลข 1-9
                    for num in range(1, 10):
                        if self.is_valid(row, col, num):
                            # ใส่ตัวเลข
                            self.board[row][col] = num
                            
                            # เรียกซ้ำเพื่อหาช่องว่างถัดไป
                            if self.solve():
                                return True
                            
                            # Backtrack: คืนค่าช่องเป็นว่าง
                            self.board[row][col] = 0
                    
                    # ไม่มีตัวเลขไหนใช้ได้ ให้ Backtrack
                    return False
        
        # ไม่มีช่องว่าง = แก้ปริศนาสำเร็จ
        return True
    
    def play(self, row, col, num):
        """
        ผู้เล่นวางตัวเลขในตำแหน่งที่กำหนด
        
        พารามิเตอร์:
        - row: แถว (0-8)
        - col: คอลัมน์ (0-8)
        - num: ตัวเลข (1-9)
        
        ตรวจสอบหลายเงื่อนไข:
        1. ตำแหน่งถูกต้อง (0-8)
        2. ตัวเลขถูกต้อง (1-9)
        3. ช่องว่างพร้อมให้เดิน
        4. ไม่ขัดแย้งกับกฎ Sudoku
        """
        # ตรวจสอบตำแหน่ง
        if not (0 <= row <= 8 and 0 <= col <= 8):
            print("❌ ตำแหน่งไม่ถูกต้อง (ต้อง 0-8)")
            return False
        
        # ตรวจสอบตัวเลข
        if not (1 <= num <= 9):
            print("❌ ตัวเลขต้องเป็น 1-9")
            return False
        
        # ตรวจสอบว่าช่องว่างหรือไม่
        if self.board[row][col] != 0:
            print(f"❌ ช่องนี้มีตัวเลข {self.board[row][col]} แล้ว")
            return False
        
        # ตรวจสอบความถูกต้องตามกฎ Sudoku
        if self.is_valid(row, col, num):
            self.board[row][col] = num
            self.moves.append((row, col, num))  # บันทึกการเดิน
            print(f"✅ วางตัวเลข {num} ที่ตำแหน่ง ({row}, {col}) สำเร็จ")
            return True
        else:
            print(f"❌ ไม่สามารถวางตัวเลข {num} ที่ตำแหน่ง ({row}, {col}) ได้ (ขัดแย้งกับกฎ Sudoku)")
            return False
    
    def is_complete(self):
        """
        ตรวจสอบว่าปริศนาเสร็จสมบูรณ์หรือไม่
        (ทุกช่องมีตัวเลข และไม่ขัดแย้ง)
        """
        # ตรวจสอบว่ามีช่องว่างหรือไม่
        for row in range(9):
            for col in range(9):
                if self.board[row][col] == 0:
                    return False
        return True
    
    def reset(self):
        """
        รีเซ็ตเกมกลับไปที่สถานะเริ่มต้น
        """
        self.board = [row[:] for row in self.initial_board]
        self.moves = []
        print("🔄 เกมถูกรีเซ็ตแล้ว")
    
    def undo(self):
        """
        ยกเลิกการเดินครั้งล่าสุด
        """
        if not self.moves:
            print("❌ ไม่มีการเดินให้ยกเลิก")
            return False
        
        row, col, _ = self.moves.pop()
        self.board[row][col] = 0
        print(f"⬅️ ยกเลิกการเดินที่ ({row}, {col})")
        return True
    
    def get_hint(self):
        """
        ให้คำใบ้โดยแสดงช่องว่างแรกที่สามารถวางตัวเลขได้
        """
        for row in range(9):
            for col in range(9):
                if self.board[row][col] == 0:
                    # ค้นหาตัวเลขที่ถูกต้องสำหรับช่องนี้
                    for num in range(1, 10):
                        if self.is_valid(row, col, num):
                            print(f"💡 คำใบ้: ที่ตำแหน่ง ({row}, {col}) สามารถใส่ {num} ได้")
                            return (row, col, num)
        
        print("💡 ไม่มีคำใบ้ (อาจเสร็จแล้ว)")
        return None
    
    def count_empty_cells(self):
        """
        นับจำนวนช่องว่างที่เหลือ
        """
        count = 0
        for row in range(9):
            for col in range(9):
                if self.board[row][col] == 0:
                    count += 1
        return count

## ส่วนที่ 3: ลองเล่นเกม - ตัวอย่างพื้นฐาน

In [ ]:
# สร้างเกมใหม่
game = SudokuGame()

print("🎮 เกม Sudoku พื้นฐาน")
print("\n📋 ตารางเดิม:")
game.display_board()
print(f"ช่องว่างที่เหลือ: {game.count_empty_cells()} ช่อง")

### ตัวอย่างการเดิน - ผู้เล่นวางตัวเลข

In [ ]:
# ผู้เล่นวางตัวเลข - ครั้งแรก (ถูกต้อง)
game.play(0, 3, 4)  # วางตัวเลข 4 ที่แถว 0 คอลัมน์ 3

In [ ]:
# วางตัวเลขครั้งที่สอง (ถูกต้อง)
game.play(0, 4, 1)  # วางตัวเลข 1 ที่แถว 0 คอลัมน์ 4

In [ ]:
# วางตัวเลขครั้งที่สาม (จะล้มเหลว - ช่องว่างแต่ปีพอ)
game.play(0, 3, 2)  # พยายามวางตัวเลข 2 ที่ตำแหน่งที่มีตัวเลข 4 แล้ว

In [ ]:
# วางตัวเลขที่ขัดแย้งกับกฎ Sudoku
game.play(0, 5, 3)  # พยายามวางตัวเลข 3 (มี 3 ในคอลัมน์ 5 แล้ว)

In [ ]:
# แสดงตารางหลังจากเล่น
print("\n📋 ตารางหลังจากเล่น:")
game.display_board()
print(f"ช่องว่างที่เหลือ: {game.count_empty_cells()} ช่อง")
print(f"เสร็จสมบูรณ์? {'ใช่ ✅' if game.is_complete() else 'ยังไม่ ❌'}")

## ส่วนที่ 4: การใช้ฟังก์ชันเพิ่มเติม

### 4.1 การให้คำใบ้

In [ ]:
# ให้คำใบ้ (hint)
print("🎮 เรียกใช้ฟังก์ชัน get_hint()")
hint = game.get_hint()

### 4.2 การยกเลิกการเดิน

In [ ]:
# ยกเลิกการเดิน
print("ยกเลิกการเดินหนึ่งครั้ง...")
game.undo()
game.display_board()
print(f"ช่องว่างที่เหลือ: {game.count_empty_cells()} ช่อง")

### 4.3 การรีเซ็ตเกม

In [ ]:
# รีเซ็ตเกม
game.reset()
print("\n📋 ตารางหลังรีเซ็ต:")
game.display_board()
print(f"ช่องว่างที่เหลือ: {game.count_empty_cells()} ช่อง")

## ส่วนที่ 5: แก้ปริศนาโดยใช้ Backtracking

In [ ]:
# สร้างเกมใหม่เพื่อแก้ปริศนา
game2 = SudokuGame()

print("ตารางเดิม:")
game2.display_board()

print("🔍 กำลังแก้ปริศนา...")
if game2.solve():
    print("✅ แก้ปริศนาสำเร็จ!\n")
    game2.display_board()
else:
    print("❌ ไม่สามารถแก้ปริศนาได้")

## ส่วนที่ 6: ตัวอย่างการสร้างเกมใหม่จากตารางที่กำหนด

In [ ]:
# ตารางปริศนา Sudoku ที่ยากขึ้น
hard_board = [
    [0, 0, 0, 0, 0, 0, 0, 1, 2],
    [0, 0, 0, 0, 3, 5, 0, 0, 0],
    [0, 0, 0, 6, 0, 0, 0, 7, 0],
    [7, 0, 0, 0, 0, 0, 3, 0, 0],
    [0, 0, 0, 4, 0, 8, 0, 0, 0],
    [0, 0, 9, 0, 0, 0, 0, 0, 8],
    [0, 5, 0, 0, 0, 9, 0, 0, 0],
    [0, 0, 0, 1, 2, 0, 0, 0, 0],
    [6, 7, 0, 0, 0, 0, 0, 0, 0]
]

# สร้างเกมจากตารางที่กำหนด
hard_game = SudokuGame(hard_board)

print("ปริศนา Sudoku (ยาก):")
hard_game.display_board()
print(f"ช่องว่างที่เหลือ: {hard_game.count_empty_cells()} ช่อง")

In [ ]:
# แก้ปริศนา
print("🔍 กำลังแก้ปริศนา...")
if hard_game.solve():
    print("✅ แก้ปริศนาสำเร็จ!\n")
    hard_game.display_board()
else:
    print("❌ ไม่สามารถแก้ปริศนาได้")

## สรุป

ในบทเรียนนี้ เราได้เรียนรู้:

### ฟังก์ชันพื้นฐาน
1. **`display_board()`** - แสดงตารางเกมอย่างสวยงาม
2. **`is_valid(row, col, num)`** - ตรวจสอบความถูกต้องตามกฎ Sudoku
3. **`solve()`** - แก้ปริศนาโดยใช้วิธี Backtracking
4. **`play(row, col, num)`** - ผู้เล่นวางตัวเลข

### ฟังก์ชันเพิ่มเติม
5. **`is_complete()`** - ตรวจสอบว่าปริศนาเสร็จแล้วหรือไม่
6. **`reset()`** - รีเซ็ตเกมกลับไปที่สถานะเริ่มต้น
7. **`undo()`** - ยกเลิกการเดินครั้งล่าสุด
8. **`get_hint()`** - ให้คำใบ้ช่องแรก
9. **`count_empty_cells()`** - นับจำนวนช่องว่างที่เหลือ

### แนวทางขยายเพิ่มเติม
- สร้างเครื่องกำเนิดปริศนา (Puzzle Generator)
- เพิ่มระดับความยากลำบาก (Easy, Medium, Hard)
- สร้างอินเทอร์เฟเซ GUI ด้วย Tkinter หรือ Pygame
- เพิ่มระบบการนับคะแนน
- บันทึกเกมและโหลดเกมที่บันทึกไว้
- เพิ่มการตรวจสอบความสามารถในการแก้ปริศนา (Solvability Check)